<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Hypothesis Testing

*Session 4 · Notebook 02 · Lecture · Coach version*

## Overview

A **hypothesis test** is a formal procedure for deciding whether a pattern in a sample is real or could simply be the result of random chance. This notebook builds the logic of hypothesis testing and then works through the **t-test** family in full: one-sample, two-sample and paired. Each test is presented with its assumptions, a complete step-by-step worked example, an effect size, and guidance on what to do when the assumptions fail.

The concepts are general purpose; a dedicated section shows how they map onto risk analysis, and the exercises put them to work on a real dataset.

## Learning Objectives

By the end of this notebook you will be able to:

- State null and alternative hypotheses and explain the p-value and significance level.
- Distinguish Type I from Type II errors and follow a process that avoids p-hacking.
- Choose and run the correct t-test (one-sample, two-sample, paired) with `scipy.stats`.
- Check every assumption of a t-test and choose a robust alternative when they fail.
- Report and interpret an effect size (Cohen's d) alongside statistical significance.

## Prerequisites

- Notebook 04_01 (sampling, standard error, confidence intervals, distributions, transformations).
- Session 2 pandas (filtering and selecting columns).

## Index

1. [Why this matters for risk analysis](#sec1)
2. [The logic of hypothesis testing](#sec2)
3. [The t-test family](#sec3)
4. [One-sample t-test: complete walkthrough](#sec4)
5. [Two-sample t-test: complete walkthrough](#sec5)
6. [Paired t-test: complete walkthrough](#sec6)
7. [Comparing categorical variables: chi-square](#secchi)
8. [When assumptions are violated](#sec7)
9. [Application: an A/B test for a process change](#sec8)
10. [Exercises](#exercises)
11. [Additional Exercises](#additional)
12. [Challenge](#challenge)
13. [Key Takeaways](#takeaways)
14. [Further Reading](#reading)

<a id="setup"></a>
# Section 0: Setup

The teaching **examples** use small, relatable datasets so each idea is self-contained. The **exercises** use the restaurant `tips` dataset that ships with seaborn (244 transactions), exactly as in the reference material.

**Documentation:** [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html) - the core library for the hypothesis tests used throughout this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
np.random.seed(42)  # reproducible results throughout

# Seaborn source (kept for reuse with other clients); to use it, swap the read_csv line for:
# tips = sns.load_dataset('tips')   # used in the exercises
# Or read directly from the public S3 bucket (no local file needed):
# tips = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_4/tips.csv')   # used in the exercises
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# tips = pd.read_csv(session_datasets_http["tips"])   # used in the exercises
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# tips = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_4/tips.csv", header=True, inferSchema=True).toPandas()   # used in the exercises
tips = pd.read_csv('../datasets/Session_4/tips.csv')   # used in the exercises
print('tips rows:', len(tips))
tips.head(3)

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Risk teams constantly have to decide whether an observed change is real or noise, and hypothesis testing is the disciplined way to do that.

| Use of hypothesis testing | Risk example |
|---|---|
| **Compare a metric to a target** | Is the realised default rate different from the 4% assumed in the model? |
| **Compare two groups** | Do two customer segments have different average loss given default? |
| **Before vs after a change** | Did a new underwriting policy change the average exposure per loan? |
| **Model and process validation** | Are this quarter's residuals consistent with last quarter's, or has the model drifted? |
| **A/B testing controls** | Did a new fraud rule actually reduce average losses, or did we get lucky? |

In every case the danger is the same: reacting to random variation as if it were a real signal. A hypothesis test, reported with an effect size, keeps decisions evidence-based.

<a id="sec2"></a>
# Section 2: The logic of hypothesis testing

**Definition:** a hypothesis test weighs two competing claims about a population using sample evidence.

- **Null hypothesis (H0):** the default, sceptical position, usually "no difference" or "no effect".
- **Alternative hypothesis (H1):** the claim we are gathering evidence for, usually "there is a difference".

**Analogy: a courtroom.** The defendant is presumed innocent (H0). The prosecution must present strong enough evidence to reject that presumption. We never "prove innocence"; we either reject the presumption or fail to reject it. A hypothesis test works the same way: we never accept H0, we only **reject** or **fail to reject** it.

### Understanding alpha and the p-value

**Alpha (the significance level)** is the threshold we set in advance for how much evidence we need.

- alpha = 0.05 (5%): most common.
- alpha = 0.01 (1%): more conservative.
- alpha = 0.10 (10%): less stringent.

It means: "we are willing to accept a 5% chance of incorrectly rejecting the null hypothesis (a false positive)".

**The p-value** is the probability of observing results as extreme as (or more extreme than) what we actually observed, **assuming the null hypothesis is true**. In plain English: "if the coffee shop's 75 mg claim is true, what is the probability of getting our sample mean just by random chance?".

| p-value | Interpretation |
|---|---|
| 0.001 | very strong evidence against H0 |
| 0.01 | strong evidence against H0 |
| 0.05 | threshold for significance |
| 0.10 | weak evidence |
| 0.50 | no evidence against H0 |

**Decision rule:**

```
if p-value <  alpha:  reject H0      (statistically significant)
if p-value >= alpha:  fail to reject H0  (not statistically significant)
```

**The p-value is NOT** the probability that H0 is true, nor the probability the result is due to chance, nor the size of the effect.

### The two kinds of error

Because we decide from a sample, we can be wrong in two ways:

| | H0 is actually true | H0 is actually false |
|---|---|---|
| **We reject H0** | Type I error (false positive), probability = alpha | Correct decision |
| **We fail to reject H0** | Correct decision | Type II error (false negative), probability = beta |

A Type I error raises a false alarm; a Type II error misses a real effect. Lowering alpha reduces false positives but increases false negatives. The **power** of a test is `1 - beta`, the chance of detecting a real effect; larger samples increase power.

### The research process (avoiding p-hacking)

**P-hacking** is manipulating data or analysis to achieve a statistically significant result. It includes:

- Collecting data and only then deciding what to test.
- Trying multiple tests until one gives p < 0.05.
- Choosing a one-tailed vs two-tailed test after seeing the results.
- Removing "inconvenient" data points.

**The correct order (decide everything BEFORE collecting data):**

```
1. Define the research question
2. State the hypotheses (H0, H1)
3. Choose the test type (one-tailed or two-tailed)
4. Set the significance level (alpha)
5. Determine the sample size
6. Register the plan (if possible)
7. THEN collect data
8. THEN analyse data
9. THEN interpret results
```

**The wrong order (p-hacking):** collect data, look at it, decide what hypothesis to test, then try different tests until something is significant.

<a id="sec3"></a>
# Section 3: The t-test family

**Definition:** a **t-test** is a statistical hypothesis test used to determine whether there is a significant difference between **means**. It is one of the most commonly used tests in research, business and science.

The fundamental question a t-test answers is:

> "Is the difference I am observing real, or could it just be due to random chance?"

**Why "t-test"?** It uses the **t-statistic** and compares it to the **t-distribution** (Student's t-distribution). The distribution was published by William Sealy Gosset under the pseudonym "Student" while he worked at the Guinness brewery in 1908.

### When do we use t-tests?

T-tests are appropriate when:

- You want to compare **means** (not medians or proportions).
- You have **continuous data** (like height, weight, test scores, temperature).
- Your sample size is **relatively small** (typically n < 30, though it also works with larger samples).
- You **do not know** the population standard deviation.

### Key concept: the logic behind t-tests

Imagine you flip a coin 10 times and get 7 heads. Is the coin biased, or could this happen by chance with a fair coin?

Similarly, if a sample mean differs from an expected value, the t-test helps determine:

1. **How different** is the sample mean from what we expected?
2. **How variable** is the data?
3. **How confident** can we be that the difference is real?

The t-test combines these three pieces of information into a single number (the t-statistic) that we can evaluate against the t-distribution.

## 3.1 The three types of t-test

### Type 1: One-sample t-test

**Purpose:** compare a sample mean to a known value or hypothesised population mean.

**Research questions:** "Does this sample differ from a standard?", "Is the average different from a claimed value?".

**Example scenarios:** testing whether an espresso contains the advertised 75 mg of caffeine; checking whether the average test score differs from the national average; verifying whether machine-filled bottles contain the labelled 500 ml.

**Hypotheses:**
```
H0: mu = mu0   (sample mean equals the hypothesised value)
H1: mu != mu0  (sample mean differs from it)
```

**Formula:** `t = (x-bar - mu0) / (s / sqrt(n))`

### Type 2: Independent two-sample t-test

**Purpose:** compare means between two separate, unrelated groups.

**Research questions:** "Do two groups differ from each other?", "Is treatment A better than treatment B?".

**Example scenarios:** comparing test scores between students who used Method A vs Method B; blood pressure in a treatment vs a control group; salaries between two companies.

**Hypotheses:**
```
H0: mu1 = mu2   (equal means)
H1: mu1 != mu2  (different means)
```

**Formula (equal variances, pooled):** `t = (x-bar1 - x-bar2) / (sp * sqrt(1/n1 + 1/n2))`, where `sp = sqrt(((n1-1)s1^2 + (n2-1)s2^2) / (n1+n2-2))`.

**Two variants:** **Student's** t-test assumes equal population variances; **Welch's** t-test does not and is the safer modern default.

### Type 3: Paired (dependent) t-test

**Purpose:** compare means from the same group measured twice, or from matched pairs.

**Research questions:** "Did the intervention cause a change?", "Is there a before-and-after difference?".

**Example scenarios:** weight before and after a diet program; student performance before and after training; left vs right hand reaction time in the same people.

**Hypotheses:**
```
H0: mu_d = 0   (mean difference is zero)
H1: mu_d != 0  (mean difference is not zero)
```

**Formula:** `t = d-bar / (sd / sqrt(n))`, where `d-bar` is the mean of the paired differences and `sd` their standard deviation.

### Comparison table

| Feature | One-Sample | Independent Two-Sample | Paired |
|---|---|---|---|
| **Groups** | 1 | 2 | 1 (measured twice) |
| **Compares** | sample vs a known value | two separate groups | before vs after |
| **Data structure** | single set of values | two separate sets | paired observations |
| **Example** | scores vs national average | men vs women heights | weight before vs after |
| **Key assumption** | normality | normality + equal variances | normality of differences |
| **Degrees of freedom** | n - 1 | n1 + n2 - 2 | n - 1 |
| **Power** | medium | lower (needs larger n) | higher (controls individual variation) |


## 3.2 Assumptions of t-tests

Before conducting any t-test you must verify that the data meets certain assumptions. Violating them can lead to incorrect conclusions.

### Assumption 1: Continuous data
The variable must be measured on a continuous scale.

- **Appropriate:** height (cm), weight (kg), temperature, test scores, time, concentration.
- **Inappropriate:** yes/no responses (use chi-square), categories (chi-square or logistic regression), ordinal rankings (non-parametric tests), count data (Poisson regression).

### Assumption 2: Independence of observations
Each observation must be independent; one measurement should not influence another.

- **Independent:** randomly selecting different people for each measurement; each participant contributes only once.
- **Dependent:** measuring the same person multiple times (use a paired test), clustered data, or time series with autocorrelation.

### Assumption 3: Normality
The data (or the sampling distribution of the mean) should be approximately normal.

- For **small samples (n < 30):** the data should be approximately normal.
- For **large samples (n >= 30):** the t-test is robust to violations thanks to the Central Limit Theorem.

**How to check:** a histogram and Q-Q plot, the Shapiro-Wilk or Kolmogorov-Smirnov test, or by inspecting skewness and kurtosis.

### Assumption 4: No significant outliers
Extreme outliers can distort results and should be investigated. Check with boxplots, z-scores (values beyond +/-3) or the IQR method (beyond Q1 - 1.5xIQR or Q3 + 1.5xIQR).

### Assumption 5: Homogeneity of variance (independent two-sample test only)
For comparing two groups the variances should be approximately equal. Check with **Levene's test** or the F-test (a rule of thumb: the ratio of variances should be between 0.5 and 2). **If violated:** use **Welch's t-test**, which does not assume equal variances, or transform the data.

## 3.3 One-tailed vs two-tailed

A **two-tailed** test asks whether the mean is *different* (higher or lower): H1 is `mu != mu0`. A **one-tailed** test asks about a specific direction: H1 is `mu > mu0` (right-tailed) or `mu < mu0` (left-tailed). Use two-tailed as the default; only use one-tailed when a directional hypothesis was decided in advance. The plot shades the rejection regions for a two-tailed test at alpha = 0.05.

In [ ]:
df_demo = 29
x = np.linspace(-4, 4, 400)
y = stats.t.pdf(x, df_demo)
crit = stats.t.ppf(0.975, df_demo)   # +/- this value cuts off 2.5% in each tail

plt.figure(figsize=(8, 4))
plt.plot(x, y, 'b-')
plt.fill_between(x, y, where=(x <= -crit), color='salmon', label='rejection region')
plt.fill_between(x, y, where=(x >= crit), color='salmon')
plt.axvline(0, color='grey', ls=':')
plt.title(f'Two-tailed test, alpha=0.05 (critical t = +/-{crit:.3f})')
plt.xlabel('t'); plt.ylabel('density'); plt.legend(); plt.show()

<a id="sec4"></a>
# Section 4: One-sample t-test - complete walkthrough

We now run a complete one-sample t-test from start to finish, following the proper process and checking every assumption.

## The Scenario: Coffee Shop Caffeine Testing

**Context:** a coffee shop advertises that their espresso shots contain **75 mg of caffeine**. As a health inspector, you want to verify this claim.

**Research Question:** does the coffee shop's espresso actually contain 75 mg of caffeine on average?

### The formula

```
        x-bar - mu0
t = ---------------
        s / sqrt(n)
```

- **Numerator (x-bar - mu0):** the difference between what we observed and what was claimed (the effect we are measuring).
- **Denominator (s / sqrt(n)):** the standard error, how much sample means typically vary, accounting for both data variability and sample size.
- **The t-statistic** is how many standard errors our sample mean sits away from the claimed value. A large |t| means a surprising result and strong evidence against H0.

### Our Example: Proper Process

Following the research process from Section 2, we decide everything **before** collecting any data:

**Step 1: Research Question (before collecting data)**  
"Does the coffee contain 75 mg of caffeine as claimed?"

**Step 2: Hypotheses (before collecting data)**  
- **H0 (null):** mu = 75 mg (the coffee meets the claim).
- **H1 (alternative):** mu != 75 mg (the coffee differs from the claim).

**Step 3: Test choice (before collecting data)**  
Two-tailed test (we want to detect caffeine that is either higher OR lower than claimed).

**Step 4: Significance level (before collecting data)**  
alpha = 0.05 (the standard 5% significance level).

**Step 5: Sample size (before collecting data)**  
We will collect 30 espresso samples (adequate for a t-test and helped by the Central Limit Theorem).

**Step 6: THEN collect data**  
Only now do we measure the caffeine content of 30 random espresso shots.

In [ ]:
caffeine = np.array([71.9, 70.1, 72.3, 74.8, 69.8, 69.8, 74.9, 72.6, 69.2, 72.0,
                     69.2, 69.2, 71.2, 65.1, 65.7, 68.9, 67.7, 71.4, 68.0, 66.5,
                     74.6, 69.9, 70.7, 66.5, 69.0, 70.8, 67.3, 71.6, 68.8, 69.7])
mu0 = 75
alpha = 0.05
print(f'n = {len(caffeine)}, sample mean = {caffeine.mean():.2f} mg')

### Step 0: Check the assumptions

**Assumption 1 (continuous):** caffeine content (mg) is continuous.  **Assumption 2 (independence):** each shot was sampled separately.  **Assumptions 3 and 4 (normality and outliers):** we check below with a boxplot, the IQR rule, a histogram, a Q-Q plot and the Shapiro-Wilk test.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].boxplot(caffeine, vert=True); ax[0].set_title('Boxplot (outliers)')
ax[0].set_ylabel('Caffeine (mg)')
ax[1].hist(caffeine, bins=10, color='steelblue', alpha=0.7); ax[1].set_title('Histogram')
stats.probplot(caffeine, dist='norm', plot=ax[2]); ax[2].set_title('Q-Q plot')
plt.tight_layout(); plt.show()

Q1, Q3 = np.percentile(caffeine, [25, 75])
IQR = Q3 - Q1
low, high = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = caffeine[(caffeine < low) | (caffeine > high)]
print(f'IQR outlier bounds: [{low:.2f}, {high:.2f}] -> '
      f'{len(outliers)} outliers detected')
print(f'Shapiro-Wilk p = {stats.shapiro(caffeine).pvalue:.3f} -> '
      f'{"normality OK" if stats.shapiro(caffeine).pvalue > 0.05 else "normality doubtful"}')

### Steps 1 to 6: calculate the t-statistic and p-value (by hand)

We follow the formula one step at a time, then confirm with scipy.

In [ ]:
x_bar = caffeine.mean()                 # Step 1: sample mean
s = caffeine.std(ddof=1)                # Step 2: sample standard deviation
n = len(caffeine)
se = s / np.sqrt(n)                     # Step 3: standard error
t_stat = (x_bar - mu0) / se            # Step 4: t-statistic
df = n - 1                              # Step 5: degrees of freedom
p_value = 2 * stats.t.sf(abs(t_stat), df)   # Step 6: two-tailed p-value

print(f'Step 1  sample mean  x-bar = {x_bar:.2f} mg')
print(f'Step 2  sample sd        s = {s:.2f} mg')
print(f'Step 3  standard error  SE = {se:.3f}')
print(f'Step 4  t-statistic      t = {t_stat:.3f}')
print(f'Step 5  degrees of freedom = {df}')
print(f'Step 6  p-value            = {p_value:.3g}')

### Confirm with scipy

In [ ]:
t_scipy, p_scipy = stats.ttest_1samp(caffeine, popmean=mu0)
print(f'scipy: t = {t_scipy:.3f}, p = {p_scipy:.3g}')

### Steps 7 and 8: decision and conclusion

In [ ]:
print('Step 7  Decision:', 'reject H0' if p_value < alpha else 'fail to reject H0')
print('Step 8  Conclusion: the sample mean of '
      f'{x_bar:.1f} mg is far below the advertised 75 mg, and the difference is '
      'statistically significant (p < 0.05). The espresso does NOT contain 75 mg of '
      'caffeine on average.')

### Effect Size: How Big is the Difference?

A p-value tells us whether there is a **statistically significant** difference, but it does not tell us **how large** the difference is in practical terms. For that we report an **effect size**. The most common one for comparing means is **Cohen's d**.

**Formula (one-sample):**

```
        x-bar - mu0
d = ---------------
             s
```

It expresses the gap between the sample mean and the claimed value in units of standard deviation. For the coffee data the sample mean is about 69.97 mg, mu0 = 75 mg and s is about 2.52 mg, so:

```
d = (69.97 - 75) / 2.52 = -5.03 / 2.52 = -2.00
```

**Interpretation (use the absolute value):**

| Cohen's d (absolute value) | Effect |
|---|---|
| around 0.2 | small |
| around 0.5 | medium |
| around 0.8 | large |
| 2.0 | very large |

A d of about -2.0 is a **very large** effect: the espresso is not just significantly below 75 mg, it is far below it in practical terms. Always report the effect size alongside the p-value, especially with large samples where even a trivial difference can be significant.

In [ ]:
cohens_d = (x_bar - mu0) / s
size = ('large' if abs(cohens_d) >= 0.8 else 'medium' if abs(cohens_d) >= 0.5
        else 'small' if abs(cohens_d) >= 0.2 else 'negligible')
print(f"Cohen's d = {cohens_d:.2f} -> {size} effect")

<a id="sec5"></a>
# Section 5: Two-sample (independent) t-test - complete walkthrough

Now a complete independent two-sample t-test, using a fun but scientifically valid study: **do video games improve reaction times?**

## The Scenario: Gamers vs Non-Gamers Reaction Time

**Context:** a psychology researcher wants to investigate whether regular video game playing is associated with faster reaction times. This has implications for training programs, hiring decisions and understanding the cognitive effects of gaming.

**Research Question:** do gamers (people who regularly play video games) have different reaction times from non-gamers?

### The Research Process (Following Best Practices)

**Step 1: Research question (before data collection)**  
"Do regular gamers have different reaction times from people who do not play?"

**Step 2: Hypotheses (before data collection)**  
- **H0:** mu_gamers = mu_non-gamers (equal reaction times).
- **H1:** mu_gamers != mu_non-gamers (different reaction times).

**Step 3: Test choice (before data collection)**  
Independent two-sample t-test, two-tailed (we want to detect a difference in either direction, though we suspect gamers may be faster).

**Step 4: Significance level (before data collection)**  
alpha = 0.05.

**Step 5: Sample size (before data collection)**  
We recruit 30 regular gamers and 30 non-gamers.

**Step 6: THEN collect data**  
Each person takes a standardised reaction-time test (visual stimulus, then button press), measured in milliseconds. Lower values mean faster reactions.

In [ ]:
gamers = np.array([221.6, 227.6, 266.7, 264.9, 292.3, 302.0, 255.6, 260.7, 232.9, 234.0,
                   174.4, 205.4, 206.1, 253.3, 233.6, 313.1, 312.0, 277.2, 277.8, 285.5,
                   241.3, 241.9, 211.0, 231.6, 284.9, 249.8, 232.2, 234.5, 248.8, 285.0])
non_gamers = np.array([232.6, 274.0, 319.6, 227.4, 338.0, 251.7, 311.8, 288.5, 299.2, 236.5,
                       289.8, 270.6, 333.0, 263.4, 284.5, 273.3, 300.6, 333.1, 285.6, 302.8,
                       243.8, 287.7, 236.5, 249.1, 334.9, 281.8, 259.5, 285.6, 229.1, 354.0])
print(f'Mean gamers = {gamers.mean():.1f} ms, mean non-gamers = {non_gamers.mean():.1f} ms')

plt.figure(figsize=(7, 4))
plt.boxplot([gamers, non_gamers], labels=['Gamers', 'Non-Gamers'])
plt.ylabel('reaction time (ms)'); plt.title('Reaction time by group'); plt.show()

### Step 0: Check the assumptions

We check normality for **both** groups (Shapiro-Wilk) and homogeneity of variance (Levene's test).

In [ ]:
print(f'Shapiro gamers     p = {stats.shapiro(gamers).pvalue:.3f}')
print(f'Shapiro non-gamers p = {stats.shapiro(non_gamers).pvalue:.3f}')
lev_p = stats.levene(gamers, non_gamers).pvalue
print(f"Levene             p = {lev_p:.3f} -> "
      f'{"equal variances OK (Student or Welch both fine)" if lev_p > 0.05 else "use Welch"}')

### Steps 1 to 6: pooled standard deviation, t-statistic and p-value (by hand)

In [ ]:
x1, x2 = gamers.mean(), non_gamers.mean()
s1, s2 = gamers.std(ddof=1), non_gamers.std(ddof=1)
n1, n2 = len(gamers), len(non_gamers)

pooled_sd = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))   # Step 2
se = pooled_sd * np.sqrt(1/n1 + 1/n2)                            # Step 3
t_stat = (x1 - x2) / se                                          # Step 4
df = n1 + n2 - 2                                                 # Step 5
p_value = 2 * stats.t.sf(abs(t_stat), df)                        # Step 6

print(f'Step 1  difference in means = {x1 - x2:.1f} ms')
print(f'Step 2  pooled sd           = {pooled_sd:.2f} ms')
print(f'Step 3  standard error      = {se:.2f}')
print(f'Step 4  t-statistic         = {t_stat:.3f}')
print(f'Step 5  degrees of freedom  = {df}')
print(f'Step 6  p-value             = {p_value:.4f}')

### Confirm with scipy (Student's and Welch's)

In [ ]:
t_student, p_student = stats.ttest_ind(gamers, non_gamers, equal_var=True)
t_welch, p_welch = stats.ttest_ind(gamers, non_gamers, equal_var=False)
print(f"Student's t: t = {t_student:.3f}, p = {p_student:.4f}")
print(f"Welch's   t: t = {t_welch:.3f}, p = {p_welch:.4f}")

### Step 7: Effect size (Cohen's d)

For two groups, Cohen's d uses the pooled standard deviation: `d = (x-bar1 - x-bar2) / sp`.

In [ ]:
cohens_d = (x1 - x2) / pooled_sd
size = ('large' if abs(cohens_d) >= 0.8 else 'medium' if abs(cohens_d) >= 0.5
        else 'small' if abs(cohens_d) >= 0.2 else 'negligible')
print(f"Cohen's d = {cohens_d:.2f} -> {size} effect")

### Step 8: Conclusion

Gamers averaged about 31 ms faster than non-gamers. The difference is statistically significant (p ~ 0.001) and the effect is large (Cohen's d ~ -0.9), so it is both real and practically meaningful.

<a id="sec6"></a>
# Section 6: Paired (dependent) t-test - complete walkthrough

Finally, a complete paired t-test, where the same units are measured twice.

## The Scenario: Weight Before and After a Diet

**Context:** a nutritionist runs a 12-week diet program and wants to know whether it works. Ten participants are weighed (kg) before starting and again at the end. Because each person is measured twice, the two sets of numbers are **paired**, not independent.

**Research Question:** does the diet program change participants' weight on average?

### The Research Process (Following Best Practices)

**Step 1: Research question (before data collection)**  
"Does the 12-week diet change weight on average?"

**Step 2: Hypotheses (before data collection)**  
- **H0:** mu_d = 0 (the diet produces no average change in weight).
- **H1:** mu_d != 0 (the diet changes weight).

**Step 3: Test choice (before data collection)**  
Paired (dependent) t-test, two-tailed.

**Step 4: Significance level (before data collection)**  
alpha = 0.05.

**Step 5: Sample size (before data collection)**  
We enrol 10 participants and measure each one twice (before and after).

**Step 6: THEN collect data**  
We record each participant's weight at the start and at the end of the program.

The paired test works on the **differences** (after minus before): it is simply a one-sample t-test asking whether the mean difference is zero.

In [ ]:
before = np.array([82, 91, 78, 85, 90, 88, 76, 95, 80, 87])
after = np.array([79, 87, 75, 84, 86, 83, 74, 90, 78, 83])
diffs = after - before
print(f'Mean before = {before.mean():.1f} kg, mean after = {after.mean():.1f} kg')
print(f'Differences (after - before): {diffs}')

### Step 0: Check the assumptions

For a paired test the key assumption is that the **differences** are approximately normal (and free of extreme outliers). We check the differences, not the raw measurements.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].boxplot(diffs, vert=True); ax[0].set_title('Boxplot of differences')
stats.probplot(diffs, dist='norm', plot=ax[1]); ax[1].set_title('Q-Q plot of differences')
plt.tight_layout(); plt.show()
print(f'Shapiro-Wilk on differences p = {stats.shapiro(diffs).pvalue:.3f} -> '
      f'{"normality OK" if stats.shapiro(diffs).pvalue > 0.05 else "normality doubtful"}')

### Steps 1 to 6: t-statistic and p-value (by hand)

In [ ]:
d_bar = diffs.mean()                 # Step 1: mean of the differences
sd = diffs.std(ddof=1)               # Step 2: sd of the differences
n = len(diffs)
se = sd / np.sqrt(n)                 # Step 3: standard error of the differences
t_stat = d_bar / se                  # Step 4: t-statistic
df = n - 1                           # Step 5: degrees of freedom
p_value = 2 * stats.t.sf(abs(t_stat), df)   # Step 6: two-tailed p-value

print(f'Step 1  mean difference  d-bar = {d_bar:.2f} kg')
print(f'Step 2  sd of differences   sd = {sd:.2f} kg')
print(f'Step 3  standard error      SE = {se:.3f}')
print(f'Step 4  t-statistic          t = {t_stat:.3f}')
print(f'Step 5  degrees of freedom     = {df}')
print(f'Step 6  p-value                = {p_value:.5f}')

### Confirm with scipy

In [ ]:
t_scipy, p_scipy = stats.ttest_rel(after, before)
print(f'scipy: t = {t_scipy:.3f}, p = {p_scipy:.5f}')

### Steps 7 and 8: effect size, decision and conclusion

For a paired design, Cohen's d uses the standard deviation of the differences: `d = d-bar / sd`.

In [ ]:
cohens_d = d_bar / sd
size = ('large' if abs(cohens_d) >= 0.8 else 'medium' if abs(cohens_d) >= 0.5
        else 'small' if abs(cohens_d) >= 0.2 else 'negligible')
print(f"Step 7  Cohen's d = {cohens_d:.2f} -> {size} effect")
print('Step 8  Decision:', 'reject H0 - the diet changed weight'
      if p_scipy < 0.05 else 'fail to reject H0')
print(f'        On average participants lost {-d_bar:.1f} kg, a statistically significant '
      'and large effect.')

<a id="secchi"></a>
# Section 7: Comparing categorical variables - the chi-square test

The t-tests so far compare the **means** of numeric data. But often both variables are **categorical** (for example, is a customer's region associated with whether they default?). For that we use the **chi-square test of independence**.

**Definition:** the chi-square test of independence checks whether two categorical variables are associated, by comparing the **observed** counts in a contingency table to the counts we would **expect** if the two were unrelated.

**Example:** testing whether the day of the week is associated with whether a meal is lunch or dinner.

**Analogy:** if knowing one variable tells you nothing about the other, they are independent. The test measures how far the real counts stray from that 'no link' expectation; the bigger the gap, the stronger the evidence of a link.

**Explanation:** build a contingency table (a cross-tab of counts), and the test sums the squared, standardised gaps between observed and expected counts into a chi-square statistic. A large statistic (small p-value) means the variables are associated.

- **H0:** the two variables are independent (not associated).
- **H1:** the two variables are associated.
- **Assumption:** the expected count in each cell should be reasonably large (a common rule of thumb is at least 5).

**Syntax:** `stats.chi2_contingency(table)` returns the chi-square statistic, the p-value, the degrees of freedom, and the table of expected counts.

### Worked example: is the day associated with the meal time?

We cross-tabulate `day` and `time` from the tips data and test whether they are independent.

In [ ]:
ct = pd.crosstab(tips['day'], tips['time'])
print('Observed counts:')
print(ct)

chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f'\nChi-square = {chi2:.1f}, dof = {dof}, p = {p:.2e}')
print('\nExpected counts if day and time were independent:')
print(pd.DataFrame(expected, index=ct.index, columns=ct.columns).round(1))

alpha = 0.05
print('\nDecision:', 'reject H0 - day and meal time are associated'
      if p < alpha else 'fail to reject H0')
print('Lunches cluster on Thur/Fri while dinners dominate the weekend, so the two are strongly linked.')

### Try it yourself

Test whether the bill payer's `sex` is associated with whether the party is a `smoker`. Build a `pd.crosstab` of the two and run `stats.chi2_contingency`. Are they associated at alpha = 0.05?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
ct = pd.crosstab(tips['sex'], tips['smoker'])
chi2, p, dof, expected = stats.chi2_contingency(ct)
print(ct)
print(f'\nChi-square = {chi2:.2f}, p = {p:.3f}')
print('reject H0: sex and smoking are associated' if p < 0.05
      else 'fail to reject H0: no significant association between sex and smoking')

<a id="sec7"></a>
# Section 8: When assumptions are violated

If the data does not meet the assumptions, you have good options rather than reporting a result you cannot trust.

**If the data is NOT normal:**

- **Transform** the data: a log transform for right-skewed data, a square-root transform for moderate skew, an inverse transform for heavy skew (see notebook 04_01).
- Use a **non-parametric** alternative, which tests medians/ranks and makes no normality assumption: the **Mann-Whitney U** test replaces the independent two-sample t-test, and the **Wilcoxon signed-rank** test replaces the paired t-test. These are slightly less powerful than a t-test when its assumptions do hold.
- Use **bootstrap** methods (resampling), which assume no specific distribution.

**If outliers are present:** investigate and only remove with clear justification (and document it); use robust methods (trimmed means, Winsorising); or transform the data.

**If independence is violated:** use a **paired t-test** for related measurements, **mixed models** for clustered data, or **time series** methods for temporal dependence.

**If variances are unequal (two-sample):** use **Welch's t-test** (`equal_var=False`).

The example below compares a skewed two-group test done three ways.

In [ ]:
# Two skewed (log-normal) groups, where the t-test's normality assumption is shaky.
np.random.seed(4)
x1 = np.random.lognormal(mean=3.00, sigma=0.7, size=40)
x2 = np.random.lognormal(mean=3.25, sigma=0.7, size=40)   # a real difference exists

p_naive = stats.ttest_ind(x1, x2, equal_var=False).pvalue
p_logged = stats.ttest_ind(np.log(x1), np.log(x2), equal_var=False).pvalue
p_mw = stats.mannwhitneyu(x1, x2, alternative='two-sided').pvalue

print(f'Naive t-test on raw data : p = {p_naive:.3f}')
print(f't-test on log data       : p = {p_logged:.3f}')
print(f'Mann-Whitney U test      : p = {p_mw:.3f}')
print('\nThe log-transform and Mann-Whitney both detect the real difference (p < 0.05), '
      'but the naive t-test on the raw skewed data misses it: the skew inflates the '
      'variance and hides the signal.')

<a id="sec8"></a>
# Section 9: Application - an A/B test for a process change

**Definition:** an **A/B test** is a controlled experiment that compares two variants, a baseline (A) and a change (B), on a chosen metric, and uses a hypothesis test to decide whether the difference between them is real or just chance. It is the two-sample t-test put to work as a decision tool.

**Example:** a team introduces a new case-handling procedure and measures the handling time (minutes) of cases under the old procedure (A) and the new one (B), with different cases in each group.

**Analogy:** a clinical trial. Patients are randomly assigned to a treatment or a placebo, and the outcomes are compared. Randomisation is what makes the comparison fair: it ensures the only systematic difference between the groups is the thing we changed.

**Explanation:** because the two groups are independent, this is a two-sample test (Welch's, the safe default). The twist is that an A/B test is a **decision**, so we report more than a p-value: an **effect size** and a **confidence interval for the difference**, because with a large enough sample even a trivial difference becomes statistically significant. We also fix the design in advance to avoid p-hacking (see the pitfalls note at the end).

## The Scenario: A New Case-Handling Procedure

**Context:** an operations team believes a redesigned procedure will speed up case handling. They run it on a batch of cases and compare the handling time against the old procedure.

**Research Question:** does the new procedure change the average handling time?

### The Research Process (Following Best Practices)

**Step 1: Research question (before data collection)**  
"Does the new procedure change average handling time?"

**Step 2: Hypotheses (before data collection)**  
- **H0:** mu_old = mu_new (the procedure makes no difference to the mean).
- **H1:** mu_old != mu_new (it changes the mean).

**Step 3: Test choice (before data collection)**  
Independent two-sample t-test, Welch's variant, two-tailed.

**Step 4: Significance level (before data collection)**  
alpha = 0.05, and we will also report Cohen's d and a 95% CI for the difference.

**Step 5: Sample size (before data collection)**  
120 cases under each procedure (large enough that the t-test is robust to non-normality).

**Step 6: THEN collect data**  
We record the handling time of every case under each procedure.

### Step 0: Collect the data and check the assumptions

With 120 cases per group the Central Limit Theorem makes the t-test robust to non-normality, so the main check is equal variances (Levene's test); if they differ, Welch's test handles it anyway.

In [ ]:
np.random.seed(7)   # reproducible for this section
old = np.random.normal(loc=12.0, scale=3.0, size=120)   # old procedure handling times (min)
new = np.random.normal(loc=11.0, scale=3.2, size=120)   # new procedure handling times (min)

print(f'Mean old = {old.mean():.2f} min, mean new = {new.mean():.2f} min')
print(f'Levene p = {stats.levene(old, new).pvalue:.3f} '
      '(unequal variances are fine because we use Welch)')

plt.figure(figsize=(7, 4))
plt.boxplot([old, new], labels=['Old procedure', 'New procedure'])
plt.ylabel('handling time (min)'); plt.title('Handling time by procedure'); plt.show()

### Steps 1 to 3: run the test, the effect size and a confidence interval

We run Welch's t-test, compute Cohen's d, and build a 95% confidence interval for the difference in means (old minus new). The CI tells us the plausible size of the improvement, not just whether one exists.

In [ ]:
# Step 1: Welch's two-sample t-test
t_stat, p_value = stats.ttest_ind(old, new, equal_var=False)

# Step 2: effect size (Cohen's d, pooled sd)
n1, n2 = len(old), len(new)
pooled_sd = np.sqrt(((n1-1)*old.var(ddof=1) + (n2-1)*new.var(ddof=1)) / (n1+n2-2))
cohens_d = (old.mean() - new.mean()) / pooled_sd

# Step 3: 95% CI for the difference in means (old - new)
diff = old.mean() - new.mean()
se_diff = np.sqrt(old.var(ddof=1)/n1 + new.var(ddof=1)/n2)
ci_lo, ci_hi = diff - 1.96*se_diff, diff + 1.96*se_diff

print(f'Difference (old - new) = {diff:.2f} min')
print(f'95% CI for the difference = [{ci_lo:.2f}, {ci_hi:.2f}] min')
print(f't = {t_stat:.3f}, p = {p_value:.4f}, Cohen\'s d = {cohens_d:.2f}')

### Step 4: decision and conclusion

We combine statistical significance (the p-value) with practical significance (the effect size and the CI) before recommending a change.

In [ ]:
alpha = 0.05
size = ('large' if abs(cohens_d) >= 0.8 else 'medium' if abs(cohens_d) >= 0.5
        else 'small' if abs(cohens_d) >= 0.2 else 'negligible')
if p_value < alpha:
    print(f'Decision: reject H0. The new procedure changes handling time '
          f'(p = {p_value:.4f}).')
    print(f'It saves about {diff:.1f} min per case on average ({size} effect, '
          f'd = {cohens_d:.2f}); the 95% CI [{ci_lo:.2f}, {ci_hi:.2f}] min shows the '
          'plausible range of the saving.')
else:
    print(f'Decision: fail to reject H0 (p = {p_value:.4f}). No evidence the procedure '
          'changes handling time.')
print('\nRecommendation: weigh the time saving against the cost of rolling out the new '
      'procedure; a statistically significant but tiny saving may not be worth it.')

### Common pitfalls in A/B testing

- **Peeking and stopping early.** Repeatedly checking the test and stopping as soon as p < 0.05 inflates the false-positive rate. Fix the sample size in advance (Step 5).
- **Testing many metrics.** The more outcomes you test, the more likely one looks significant by chance; correct for multiple comparisons or pre-register the primary metric.
- **Ignoring practical significance.** Always report the effect size and CI, not just the p-value.
- **Broken randomisation.** If the two groups differ for some reason other than the change (for example, the new procedure was only used on easy cases), the comparison is confounded and the test is meaningless.

<a id="exercises"></a>
# Section 10: Exercises - Restaurant Tips Dataset

All exercises use the `tips` dataset loaded in Section 0 (244 restaurant transactions: `total_bill`, `tip`, `sex`, `smoker`, `day`, `time`, `size`). For each test: state the hypotheses, check the assumptions, run the appropriate test, compute the effect size and write a short conclusion. Use alpha = 0.05.

### Exercise 1: One-sample t-test - tip percentage

Industry guidance suggests customers tip about **15%** of the bill. Test whether this restaurant's tip percentage (`tip / total_bill`) differs from 15%. Run a one-sample t-test and compute Cohen's d.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
tips['tip_pct'] = tips['tip'] / tips['total_bill']
t, p = stats.ttest_1samp(tips['tip_pct'], popmean=0.15)
d = (tips['tip_pct'].mean() - 0.15) / tips['tip_pct'].std()
print(f"Sample mean tip % = {tips['tip_pct'].mean()*100:.2f}%")
print(f't = {t:.3f}, p = {p:.5f}, Cohen\'s d = {d:.2f}')
print('reject H0: tip % differs from 15%' if p < 0.05 else 'fail to reject H0')

### Exercise 2: Two-sample t-test - male vs female customers

Is there a difference in the `tip` amount left by male and female bill payers? Split by `sex`, check equal variances with Levene's test, run Welch's t-test and compute Cohen's d.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
male = tips.loc[tips['sex'] == 'Male', 'tip']
female = tips.loc[tips['sex'] == 'Female', 'tip']
print(f'Levene p = {stats.levene(male, female).pvalue:.3f}')
t, p = stats.ttest_ind(male, female, equal_var=False)
n1, n2 = len(male), len(female)
pooled = np.sqrt(((n1-1)*male.var(ddof=1) + (n2-1)*female.var(ddof=1)) / (n1+n2-2))
d = (male.mean() - female.mean()) / pooled
print(f'Mean male = {male.mean():.2f}, mean female = {female.mean():.2f}')
print(f't = {t:.3f}, p = {p:.4f}, Cohen\'s d = {d:.2f}')
print('reject H0' if p < 0.05 else 'fail to reject H0 (no significant difference)')

<a id="additional"></a>
## Additional Exercises

### Exercise 3: Two-sample t-test - smokers vs non-smokers

Does smoking status affect tipping? Compare `tip` amounts between tables with smokers (`smoker == 'Yes'`) and without (`smoker == 'No'`). Run Welch's t-test and compute Cohen's d.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
smoker = tips.loc[tips['smoker'] == 'Yes', 'tip']
nonsmoker = tips.loc[tips['smoker'] == 'No', 'tip']
t, p = stats.ttest_ind(smoker, nonsmoker, equal_var=False)
n1, n2 = len(smoker), len(nonsmoker)
pooled = np.sqrt(((n1-1)*smoker.var(ddof=1) + (n2-1)*nonsmoker.var(ddof=1)) / (n1+n2-2))
d = (smoker.mean() - nonsmoker.mean()) / pooled
print(f'Mean smoker = {smoker.mean():.2f}, mean non-smoker = {nonsmoker.mean():.2f}')
print(f't = {t:.3f}, p = {p:.4f}, Cohen\'s d = {d:.2f}')
print('reject H0' if p < 0.05 else 'fail to reject H0 (no significant difference)')

<a id="challenge"></a>
## Challenge (optional): Exercise 4 - lunch vs dinner

Restaurant managers want to know if customers tip differently at **lunch** versus **dinner**, to inform staffing and shift premiums. Run the full workflow: state the hypotheses, check the assumptions (normality and equal variances), run the appropriate two-sample test, compute Cohen's d, and write a conclusion that includes the business implication. The coach answer shows one complete solution.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
lunch = tips.loc[tips['time'] == 'Lunch', 'tip']
dinner = tips.loc[tips['time'] == 'Dinner', 'tip']

# Assumptions: normality (large-ish n, so the t-test is fairly robust) and equal variances
print(f'n lunch = {len(lunch)}, n dinner = {len(dinner)}')
print(f'Levene p = {stats.levene(lunch, dinner).pvalue:.3f}')

# Welch's t-test (safe default)
t, p = stats.ttest_ind(lunch, dinner, equal_var=False)
n1, n2 = len(lunch), len(dinner)
pooled = np.sqrt(((n1-1)*lunch.var(ddof=1) + (n2-1)*dinner.var(ddof=1)) / (n1+n2-2))
d = (lunch.mean() - dinner.mean()) / pooled
print(f'Mean lunch = {lunch.mean():.2f}, mean dinner = {dinner.mean():.2f}')
print(f't = {t:.3f}, p = {p:.4f}, Cohen\'s d = {d:.2f}')

if p < 0.05:
    print('\nConclusion: reject H0. Tips differ significantly between lunch and dinner.')
else:
    print('\nConclusion: fail to reject H0. There is no significant difference in tip '
          'amount between lunch and dinner, and Cohen\'s d confirms any gap is tiny. '
          'Business implication: a shift premium based on tip size is not justified by this data.')

<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| H0 / H1 | Null (no effect) vs alternative (an effect) hypotheses |
| alpha, p-value | Threshold set in advance; p < alpha means reject H0 |
| Type I / Type II error | False positive (alpha) / false negative (beta) |
| `stats.ttest_1samp(x, popmean)` | One-sample t-test (sample vs a value) |
| `stats.ttest_ind(a, b, equal_var=False)` | Two-sample t-test; `equal_var=False` is Welch's |
| `stats.ttest_rel(after, before)` | Paired t-test (same units, two measurements) |
| `stats.shapiro`, `stats.levene` | Check normality and equal variances |
| Cohen's d | Effect size: how big the difference is, in standard deviations |
| `stats.mannwhitneyu`, `stats.wilcoxon` | Non-parametric alternatives when normality fails |
| `stats.chi2_contingency(table)` | Chi-square test of independence (two categorical variables) |


## Conclusion

You can now state hypotheses, choose and run the right t-test through a full step-by-step workflow, check every assumption, report an effect size, and fall back to a robust alternative when the assumptions fail. The next notebook turns to statistical approaches for data quality: outliers, missing-data mechanisms and imputation.

<a id="reading"></a>
## Further Reading & Resources

- [SciPy stats: statistical tests](https://docs.scipy.org/doc/scipy/reference/stats.html) the full catalogue of tests.
- [Penn State STAT 500: Hypothesis Testing](https://online.stat.psu.edu/stat500/) a clear, worked refresher.
- [GraphPad t-test calculator](https://www.graphpad.com/quickcalcs/ttest1/) check your results.
- Cohen, J. (1988). *Statistical Power Analysis for the Behavioral Sciences* the source of the effect-size benchmarks.